# 01 — Exploração do banco `cinerocket.db`

Objetivo: conhecer a camada Gold da CineData antes de construir o agente Text-to-SQL —
tabelas, ligações, qualidade dos dados e **armadilhas** que viram regras de negócio.

> O banco é aberto **somente para leitura** (`mode=ro`). Nenhuma célula altera dados.

## 0. Configuração
Abrimos a conexão em modo somente leitura e criamos um atalho `q()` que executa SQL e devolve um DataFrame.

In [1]:
import re
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

# mode=ro + uri=True: o SQLite recusa qualquer escrita nesta conexão
conn = sqlite3.connect("file:../cinerocket.db?mode=ro", uri=True)


def q(sql: str) -> pd.DataFrame:
    """Executa uma consulta e devolve um DataFrame."""
    return pd.read_sql_query(sql, conn)


q("SELECT sqlite_version() AS versao_sqlite")

,versao_sqlite
0,3.50.4


## 1. Visão geral das tabelas
Quais tabelas existem e quantas linhas cada uma tem. A tabela `alembic_version` é controle de migração, não dado de negócio.

In [2]:
tabelas_todas = q("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")["name"].tolist()
print("Todas as tabelas:", tabelas_todas)

TABELAS = [t for t in tabelas_todas if t != "alembic_version" and not t.startswith("sqlite_")]
resumo = pd.DataFrame(
    {"tabela": TABELAS, "linhas": [q(f"SELECT COUNT(*) AS n FROM {t}")["n"][0] for t in TABELAS]}
)
resumo

Todas as tabelas: ['alembic_version', 'bridge_movie_company', 'bridge_movie_genre', 'bridge_movie_person', 'dim_companies', 'dim_genres', 'dim_movies', 'dim_people', 'dim_reviews', 'fact_movies_performance', 'movie_reviews']


,tabela,linhas
0,bridge_movie_company,116326
1,bridge_movie_genre,121521
2,bridge_movie_person,745450
3,dim_companies,45941
4,dim_genres,19
5,dim_movies,95645
6,dim_people,424656
7,dim_reviews,40267
8,fact_movies_performance,95645
9,movie_reviews,43666


### 1.1 Colunas, tipos e 3 linhas de exemplo
Para cada tabela: estrutura (`PRAGMA table_info`) e uma amostra.

In [3]:
for t in TABELAS:
    print("=" * 100)
    print(f"TABELA: {t}")
    info = q(f"PRAGMA table_info({t})")[["name", "type", "notnull", "pk"]]
    display(info)
    display(q(f"SELECT * FROM {t} LIMIT 3"))

TABELA: bridge_movie_company


,name,type,notnull,pk
0,sk_movie_id,VARCHAR(64),1,1
1,sk_company_id,VARCHAR(64),1,2


,sk_movie_id,sk_company_id
0,8a85beff7b0f64692b227466b2c9c4abb274cf1225da4d5ce495da2180ba89c8,a93d6d918018600b5fba9d39dbb7898430ece8d5fb29e42a2f8db59a37f5faf7
1,c1e53d82aeac91f16040913257678ba690813daa7a5c5d3e182866a339553ce0,f0ed91f80406391e38f2e7cc647c452f8f5919e770d7d1247969c27f14bc0980
2,ceeda10da2fb7789a0c63741e562f050cec19f48ab727861678a10e51f9fc1f9,e8170686c11431164d3ad11ef91043bfb5a67a6a4af9f7e9f6e3ea51476398bd


TABELA: bridge_movie_genre


,name,type,notnull,pk
0,sk_movie_id,VARCHAR(64),1,1
1,sk_genre_id,VARCHAR(64),1,2


...

,name,type,notnull,pk
0,sk_movie_id,VARCHAR(64),1,1
1,orcamento_usd,"NUMERIC(18, 2)",0,0
2,receita_usd,"NUMERIC(18, 2)",0,0
3,lucro_usd,"NUMERIC(18, 2)",1,0
4,orcamento_brl,"NUMERIC(18, 2)",0,0
5,receita_brl,"NUMERIC(18, 2)",0,0
6,lucro_brl,"NUMERIC(18, 2)",1,0
7,popularidade,DOUBLE,0,0
8,nota_tmdb,DOUBLE,0,0
9,qtd_tmdb,INTEGER,0,0


,sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_tmdb,qtd_tmdb,nota_imdb,qtd_imdb
0,b5c312c4d8cee94972412b9ea785976f9dbb2c26dd11480d9f2b9b4691df1571,25000000.0,83080890.0,58080890,78682500.0,261480485.1,182797985.1,24.584,4.966,2375,4.5,46286.0
1,d286f479f6764d7a09951df2c51cc95b7615c4f021fedc31773b69fa3e4da5f1,NaN,NaN,0,NaN,NaN,0.0,8.929,7.064,118,6.6,4617.0
2,54ecb594e9ce04d4b5ba09215e8d130683c7fe6d8ef5f0d469691ae1dfe6df62,7500000.0,NaN,-7500000,32363250.0,NaN,-32363250.0,NaN,3.161,28,NaN,NaN


TABELA: movie_reviews


,name,type,notnull,pk
0,id,INTEGER,1,1
1,sk_movie_review_id,VARCHAR(64),1,0
2,sk_movie_id,VARCHAR(64),1,0
3,name,VARCHAR(120),1,0
4,rating,DOUBLE,1,0
5,text,VARCHAR(4000),1,0
6,created_at,DATETIME,1,0


,id,sk_movie_review_id,sk_movie_id,name,rating,text,created_at
0,1,a9605da4b01e5f61d0623ddf0ff189de61e1ddd9dbddd30213ed609c2bbaf54e,48767763d50f8bdac4e2c2af298a83562e4f0031de6d2446ccb10bac86d8eacc,Henrique Carvalho,9.8,"Adorei cada minuto, uma experiência inesquecível.",2026-09-28 15:49:34
1,2,a8e08425791b3b5d7be3c73842e2ccf747d274baf2819ace31081e2853c5ebda,f74334b36599b97ff4eb7fd0bb76d946b9df5e711062b73885f742fec7c1c9f7,Lucas Silva,2.4,Horrível! Perda de tempo.,2026-09-28 15:49:34
2,3,4bf79b3c90409e763565e1dc8eb28462ba895bfaf1bbe2acd2403f194fab1701,7f81b30a8987982c36276f599aec2e07bceecd2c7bfd68b2136fe6bfbfa500c5,Gabriela Cardoso,7.7,"Ótimo entretenimento, não decepciona.",2026-09-28 15:49:34


## 2. Como as tabelas se ligam
Primeiro procuramos chaves estrangeiras declaradas; depois, colunas com o mesmo nome em mais de uma tabela.

In [4]:
fks = []
for t in TABELAS:
    df = q(f"PRAGMA foreign_key_list({t})")
    if not df.empty:
        df.insert(0, "tabela", t)
        fks.append(df[["tabela", "from", "table", "to"]])
pd.concat(fks, ignore_index=True) if fks else "Nenhuma chave estrangeira declarada."

,tabela,from,table,to
0,bridge_movie_company,sk_company_id,dim_companies,sk_company_id
1,bridge_movie_company,sk_movie_id,dim_movies,sk_movie_id
2,bridge_movie_genre,sk_genre_id,dim_genres,sk_genre_id
3,bridge_movie_genre,sk_movie_id,dim_movies,sk_movie_id
4,bridge_movie_person,sk_person_id,dim_people,sk_person_id
5,bridge_movie_person,sk_movie_id,dim_movies,sk_movie_id
6,dim_reviews,sk_movie_id,dim_movies,sk_movie_id
7,fact_movies_performance,sk_movie_id,dim_movies,sk_movie_id
8,movie_reviews,sk_movie_id,dim_movies,sk_movie_id


In [5]:
colunas = pd.concat(
    [q(f"PRAGMA table_info({t})")[["name"]].assign(tabela=t) for t in TABELAS], ignore_index=True
)
comuns = colunas.groupby("name")["tabela"].apply(list)
comuns = comuns[comuns.str.len() > 1]
comuns.to_frame("tabelas_com_a_coluna")

,tabelas_com_a_coluna
name,
sk_company_id,"[bridge_movie_company, dim_companies]"
sk_genre_id,"[bridge_movie_genre, dim_genres]"
sk_movie_id,"[bridge_movie_company, bridge_movie_genre, bridge_movie_person, dim_movies, ..."
sk_person_id,"[bridge_movie_person, dim_people]"


### 2.1 Integridade das ligações
Existem `sk_movie_id` (ou outras chaves) apontando para registros que não existem? A fact tem uma linha por filme?

In [6]:
checagens = {
    "fact -> dim_movies": "SELECT COUNT(*) FROM fact_movies_performance f LEFT JOIN dim_movies m USING (sk_movie_id) WHERE m.sk_movie_id IS NULL",
    "bridge_genre -> dim_movies": "SELECT COUNT(*) FROM bridge_movie_genre b LEFT JOIN dim_movies m USING (sk_movie_id) WHERE m.sk_movie_id IS NULL",
    "bridge_genre -> dim_genres": "SELECT COUNT(*) FROM bridge_movie_genre b LEFT JOIN dim_genres g USING (sk_genre_id) WHERE g.sk_genre_id IS NULL",
    "bridge_person -> dim_movies": "SELECT COUNT(*) FROM bridge_movie_person b LEFT JOIN dim_movies m USING (sk_movie_id) WHERE m.sk_movie_id IS NULL",
    "bridge_person -> dim_people": "SELECT COUNT(*) FROM bridge_movie_person b LEFT JOIN dim_people p USING (sk_person_id) WHERE p.sk_person_id IS NULL",
    "bridge_company -> dim_movies": "SELECT COUNT(*) FROM bridge_movie_company b LEFT JOIN dim_movies m USING (sk_movie_id) WHERE m.sk_movie_id IS NULL",
    "bridge_company -> dim_companies": "SELECT COUNT(*) FROM bridge_movie_company b LEFT JOIN dim_companies c USING (sk_company_id) WHERE c.sk_company_id IS NULL",
    "dim_reviews -> dim_movies": "SELECT COUNT(*) FROM dim_reviews r LEFT JOIN dim_movies m USING (sk_movie_id) WHERE m.sk_movie_id IS NULL",
    "movie_reviews -> dim_movies": "SELECT COUNT(*) FROM movie_reviews r LEFT JOIN dim_movies m USING (sk_movie_id) WHERE m.sk_movie_id IS NULL",
}
orfaos = pd.DataFrame(
    [(k, conn.execute(v).fetchone()[0]) for k, v in checagens.items()], columns=["ligacao", "orfaos"]
)
display(orfaos)

q("""
SELECT
  (SELECT COUNT(*) FROM dim_movies)                                AS filmes_dim,
  (SELECT COUNT(*) FROM fact_movies_performance)                   AS linhas_fact,
  (SELECT COUNT(DISTINCT sk_movie_id) FROM fact_movies_performance) AS filmes_distintos_fact,
  (SELECT COUNT(*) FROM dim_reviews)                               AS linhas_dim_reviews,
  (SELECT COUNT(DISTINCT sk_movie_id) FROM dim_reviews)            AS filmes_dim_reviews,
  (SELECT COUNT(DISTINCT id_filme) FROM dim_movies)                AS id_filme_distintos
""")

,ligacao,orfaos
0,fact -> dim_movies,0
1,bridge_genre -> dim_movies,0
2,bridge_genre -> dim_genres,0
3,bridge_person -> dim_movies,0
4,bridge_person -> dim_people,0
5,bridge_company -> dim_movies,0
6,bridge_company -> dim_companies,0
7,dim_reviews -> dim_movies,0
8,movie_reviews -> dim_movies,0


,filmes_dim,linhas_fact,filmes_distintos_fact,linhas_dim_reviews,filmes_dim_reviews,id_filme_distintos
0,95645,95645,95645,40267,40267,95645


## 3. Perfil das colunas numéricas
Para `fact_movies_performance` e `dim_movies`: % de nulos, % de zeros, mínimo, máximo e média.

In [7]:
def perfil_numerico(tabela: str) -> pd.DataFrame:
    """Resumo de nulos, zeros e estatísticas das colunas numéricas de uma tabela."""
    df = q(f"SELECT * FROM {tabela}")
    linhas = []
    for c in df.columns:
        s = pd.to_numeric(df[c], errors="coerce")
        # só considera numérica se os valores não nulos originais viraram número
        if df[c].notna().sum() == 0 or s.notna().sum() < df[c].notna().sum():
            continue
        linhas.append({
            "coluna": c,
            "% nulos": round(100 * df[c].isna().mean(), 2),
            "% zeros": round(100 * (s == 0).mean(), 2),
            "min": s.min(),
            "max": s.max(),
            "media": round(s.mean(), 2),
        })
    return pd.DataFrame(linhas)


print("fact_movies_performance")
display(perfil_numerico("fact_movies_performance"))
print("dim_movies")
display(perfil_numerico("dim_movies"))

fact_movies_performance


,coluna,% nulos,% zeros,min,max,media
0,orcamento_usd,91.71,0.00,1.000000e+00,8.880000e+08,8.730889e+06
1,receita_usd,96.47,0.00,1.000000e+00,2.800000e+09,5.294031e+07
2,lucro_usd,0.00,89.95,-8.880000e+08,2.444000e+09,1.143464e+06
3,orcamento_brl,91.71,0.00,3.190000e+00,4.371002e+09,3.715707e+07
4,receita_brl,96.47,0.00,3.090000e+00,1.239014e+10,2.175225e+08
5,lucro_brl,0.00,89.95,-4.371002e+09,9.933737e+09,4.591946e+06
6,popularidade,3.78,1.78,0.000000e+00,2.994357e+03,3.230000e+00
7,nota_tmdb,0.00,37.83,0.000000e+00,1.000000e+01,3.820000e+00
8,qtd_tmdb,7.91,34.72,0.000000e+00,2.889400e+04,5.668000e+01
9,nota_imdb,13.25,0.00,0.000000e+00,1.000000e+01,6.270000e+00


dim_movies


,coluna,% nulos,% zeros,min,max,media
0,id_filme,0.0,0.00,14564,1760967,736724.60
1,ano_lancamento,0.0,0.00,2016,2029,2019.39
2,duracao_minutos,0.0,10.62,0,13319,67.29


## 4. Armadilhas

### 4.1 Receita e orçamento: zerados ou nulos?
Quantos filmes têm receita/orçamento realmente informados (> 0)?

In [8]:
cols_fin = ["orcamento_usd", "receita_usd", "lucro_usd", "orcamento_brl", "receita_brl", "lucro_brl"]
partes = []
for c in cols_fin:
    partes.append(q(f"""
        SELECT '{c}' AS coluna,
               SUM({c} IS NULL) AS nulos,
               SUM({c} = 0)     AS zeros,
               SUM({c} > 0)     AS positivos,
               SUM({c} < 0)     AS negativos,
               COUNT(*)         AS total
        FROM fact_movies_performance"""))
fin = pd.concat(partes, ignore_index=True)
fin["% informado (>0)"] = (100 * fin["positivos"] / fin["total"]).round(2)
fin

,coluna,nulos,zeros,positivos,negativos,total,% informado (>0)
0,orcamento_usd,87719,0,7926,0,95645,8.29
1,receita_usd,92272,0,3373,0,95645,3.53
2,lucro_usd,0,86032,2736,6877,95645,2.86
3,orcamento_brl,87719,0,7926,0,95645,8.29
4,receita_brl,92272,0,3373,0,95645,3.53
5,lucro_brl,0,86032,2736,6877,95645,2.86


In [9]:
q("""
SELECT
  SUM(receita_usd > 0)                       AS receita_informada,
  SUM(orcamento_usd > 0)                     AS orcamento_informado,
  SUM(receita_usd > 0 AND orcamento_usd > 0) AS ambos_informados,
  SUM(COALESCE(receita_usd,0) = 0 AND COALESCE(orcamento_usd,0) = 0) AS nenhum_informado,
  COUNT(*)                                   AS total
FROM fact_movies_performance
""")

,receita_informada,orcamento_informado,ambos_informados,nenhum_informado,total
0,3373,7926,1630,85976,95645


### 4.2 Lucro: é sempre receita − orçamento?
Verificamos a fórmula em todas as linhas (USD e BRL) e olhamos os filmes com orçamento informado mas receita 0/nula —
o “lucro” deles tende a ser negativo e enganoso.

In [10]:
q("""
SELECT
  COUNT(*) AS total,
  SUM(lucro_usd IS NULL) AS lucro_usd_nulo,
  SUM(ABS(COALESCE(lucro_usd,0) - (COALESCE(receita_usd,0) - COALESCE(orcamento_usd,0))) > 1) AS divergencias_usd,
  SUM(lucro_brl IS NULL) AS lucro_brl_nulo,
  SUM(ABS(COALESCE(lucro_brl,0) - (COALESCE(receita_brl,0) - COALESCE(orcamento_brl,0))) > 1) AS divergencias_brl
FROM fact_movies_performance
""")

,total,lucro_usd_nulo,divergencias_usd,lucro_brl_nulo,divergencias_brl
0,95645,0,0,0,0


In [11]:
display(q("""
SELECT
  COUNT(*) AS receita_zero_ou_nula_com_orcamento,
  SUM(lucro_usd < 0) AS lucro_negativo,
  SUM(lucro_usd = -orcamento_usd) AS lucro_igual_menos_orcamento,
  MIN(lucro_usd) AS menor_lucro_usd
FROM fact_movies_performance
WHERE COALESCE(receita_usd, 0) = 0 AND orcamento_usd > 0
"""))

q("""
SELECT m.titulo, m.ano_lancamento, f.orcamento_usd, f.receita_usd, f.lucro_usd
FROM fact_movies_performance f JOIN dim_movies m USING (sk_movie_id)
WHERE COALESCE(f.receita_usd, 0) = 0 AND f.orcamento_usd > 0
ORDER BY f.orcamento_usd DESC
LIMIT 10
""")

,receita_zero_ou_nula_com_orcamento,lucro_negativo,lucro_igual_menos_orcamento,menor_lucro_usd
0,6296,6296,6296,-888000000


,titulo,ano_lancamento,orcamento_usd,receita_usd,lucro_usd
0,Enea,2023,888000000,None,-888000000
1,Red Dead Redemption 2,2018,500000000,None,-500000000
2,The Marvels,2023,274800000,None,-274800000
3,Guardians Of The Galaxy Vol. 3,2023,250000000,None,-250000000
4,Fast & Furious Presents: Hobbs & Shaw,2019,200000000,None,-200000000
5,Wish,2023,200000000,None,-200000000
6,"Ninguém Entra, Ninguém Sai",2017,193274344,None,-193274344
7,The Batman,2022,185000000,None,-185000000
8,Miss Teacher,2016,180000000,None,-180000000
9,Mowgli: Legend Of The Jungle,2018,175000000,None,-175000000


E o caso oposto: receita informada mas orçamento zero — o lucro fica igual à receita (inflado).

In [12]:
q("""
SELECT COUNT(*) AS receita_sem_orcamento,
       SUM(lucro_usd = receita_usd) AS lucro_igual_receita
FROM fact_movies_performance
WHERE receita_usd > 0 AND COALESCE(orcamento_usd, 0) = 0
""")

,receita_sem_orcamento,lucro_igual_receita
0,1743,1743


### 4.3 Moeda: existe valor em R$?
Há colunas `_brl`. Verificamos se a conversão usa uma taxa fixa (razão BRL/USD).

In [13]:
taxa = q("""
SELECT receita_brl * 1.0 / receita_usd   AS taxa_receita,
       orcamento_brl * 1.0 / orcamento_usd AS taxa_orcamento
FROM fact_movies_performance
WHERE receita_usd > 0 AND orcamento_usd > 0
""")
taxa.describe().round(4)

,taxa_receita,taxa_orcamento
count,1630.0000,1630.0000
mean,4.0564,4.0564
std,0.8339,0.8339
min,3.0504,3.0504
25%,3.2817,3.2818
50%,3.8646,3.8646
75%,4.9480,4.9480
max,5.7797,5.7797


### 4.4 Notas: escala de TMDB, IMDb e das avaliações de usuários
Comparamos mínimo, máximo, média e quantos zeros vêm de filmes sem votos.

In [14]:
q("""
SELECT 'nota_tmdb' AS nota, MIN(nota_tmdb) AS min, MAX(nota_tmdb) AS max, ROUND(AVG(nota_tmdb),2) AS media,
       SUM(nota_tmdb IS NULL) AS nulos, SUM(nota_tmdb = 0) AS zeros, SUM(nota_tmdb = 0 AND COALESCE(qtd_tmdb,0) = 0) AS zeros_sem_votos
FROM fact_movies_performance
UNION ALL
SELECT 'nota_imdb', MIN(nota_imdb), MAX(nota_imdb), ROUND(AVG(nota_imdb),2),
       SUM(nota_imdb IS NULL), SUM(nota_imdb = 0), SUM(nota_imdb = 0 AND COALESCE(qtd_imdb,0) = 0)
FROM fact_movies_performance
UNION ALL
SELECT 'dim_reviews.nota_media_usuarios', MIN(nota_media_usuarios), MAX(nota_media_usuarios), ROUND(AVG(nota_media_usuarios),2),
       SUM(nota_media_usuarios IS NULL), SUM(nota_media_usuarios = 0), SUM(nota_media_usuarios = 0 AND COALESCE(qtd_avaliacoes_usuarios,0) = 0)
FROM dim_reviews
UNION ALL
SELECT 'movie_reviews.rating', MIN(rating), MAX(rating), ROUND(AVG(rating),2),
       SUM(rating IS NULL), SUM(rating = 0), NULL
FROM movie_reviews
""")

,nota,min,max,media,nulos,zeros,zeros_sem_votos
0,nota_tmdb,0.0,10.0,3.82,0,36185,36043.0
1,nota_imdb,0.0,10.0,6.27,12674,2,2.0
2,dim_reviews.nota_media_usuarios,0.0,10.0,5.01,0,164,0.0
3,movie_reviews.rating,0.0,10.0,5.00,0,202,NaN


In [15]:
dist = q("SELECT ROUND(nota_tmdb) AS faixa, COUNT(*) AS tmdb FROM fact_movies_performance GROUP BY 1").merge(
    q("SELECT ROUND(nota_imdb) AS faixa, COUNT(*) AS imdb FROM fact_movies_performance GROUP BY 1"),
    on="faixa", how="outer",
)
display(dist.sort_values("faixa"))
q("SELECT rating, COUNT(*) AS n FROM movie_reviews GROUP BY rating ORDER BY rating")

,faixa,tmdb,imdb
0,0.0,36185.0,2
1,1.0,965.0,96
2,2.0,1782.0,652
3,3.0,1585.0,2457
4,4.0,3944.0,5554
5,5.0,9011.0,11717
6,6.0,16058.0,22168
7,7.0,14625.0,24921
8,8.0,6891.0,11914
9,9.0,1974.0,3198


,rating,n
0,0.00,202
1,0.10,409
2,0.15,2
3,0.20,425
4,0.30,415
...,...,...
286,9.77,1
287,9.80,404
288,9.85,1
289,9.90,410


### 4.5 Popularidade
Qual a escala da coluna `popularidade`, quantos nulos/zeros e quais os filmes mais populares?

In [16]:
pop = q("SELECT popularidade FROM fact_movies_performance")["popularidade"]
print("nulos:", pop.isna().sum(), "| zeros:", (pop == 0).sum())
display(pop.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).round(3).to_frame())
q("""
SELECT m.titulo, m.ano_lancamento, f.popularidade, f.qtd_tmdb
FROM fact_movies_performance f JOIN dim_movies m USING (sk_movie_id)
ORDER BY f.popularidade DESC LIMIT 5
""")

nulos: 3615 | zeros: 1698


,popularidade
count,92030.000
mean,3.229
std,26.327
min,0.000
25%,0.600
50%,0.917
75%,1.991
90%,5.200
99%,30.681
max,2994.357


,titulo,ano_lancamento,popularidade,qtd_tmdb
0,Blue Beetle,2023,2994.357,1023
1,Gran Turismo,2023,2680.593,702
2,La Fellinette,2020,2020.000,2
3,The Fear Footage 2: Curse Of The Tape,2020,2019.000,5
4,Wwe Survivor Series 2018,2018,2018.000,17


### 4.6 Ator × diretor
`bridge_movie_person` só tem as chaves. O papel da pessoa fica em `dim_people.tipo_pessoa` — vemos os valores.

In [17]:
display(q("SELECT tipo_pessoa, COUNT(*) AS pessoas FROM dim_people GROUP BY 1 ORDER BY 2 DESC"))
display(q("""
SELECT p.tipo_pessoa, COUNT(*) AS vinculos, COUNT(DISTINCT b.sk_movie_id) AS filmes
FROM bridge_movie_person b JOIN dim_people p USING (sk_person_id)
GROUP BY 1 ORDER BY 2 DESC
"""))
q("""
SELECT nome_pessoa, COUNT(*) AS registros, GROUP_CONCAT(tipo_pessoa) AS tipos
FROM dim_people GROUP BY nome_pessoa HAVING COUNT(*) > 1
ORDER BY registros DESC LIMIT 10
""")

,tipo_pessoa,pessoas
0,Ator,273400
1,Roteirista,86056
2,Diretor,65200


,tipo_pessoa,vinculos,filmes
0,Ator,528617,80593
1,Roteirista,122621,70173
2,Diretor,94212,79464


,nome_pessoa,registros,tipos
0,0.6,3,"Ator,Diretor,Roteirista"
1,1.4,3,"Ator,Diretor,Roteirista"
2,Aaqib Khan,3,"Ator,Diretor,Roteirista"
3,Aaron Abrams,3,"Ator,Diretor,Roteirista"
4,Aaron Arens,3,"Ator,Diretor,Roteirista"
5,Aaron Cohen,3,"Ator,Diretor,Roteirista"
6,Aaron Dunbar,3,"Ator,Diretor,Roteirista"
7,Aaron Garrett,3,"Ator,Diretor,Roteirista"
8,Aaron James,3,"Ator,Diretor,Roteirista"
9,Aaron Jay Rome,3,"Ator,Diretor,Roteirista"


In [18]:
q("""
SELECT COUNT(*) AS nomes_com_mais_de_um_registro,
       SUM(n_tipos > 1) AS nomes_com_mais_de_um_tipo
FROM (SELECT nome_pessoa, COUNT(*) AS n, COUNT(DISTINCT tipo_pessoa) AS n_tipos
      FROM dim_people GROUP BY nome_pessoa HAVING COUNT(*) > 1)
""")

,nomes_com_mais_de_um_registro,nomes_com_mais_de_um_tipo
0,48210,48210


### 4.7 `dim_reviews` × `movie_reviews`
Uma parece ser o agregado por filme e a outra as avaliações individuais. Conferimos cobertura e se as médias batem.

In [19]:
display(q("""
SELECT 'dim_reviews' AS tabela, COUNT(*) AS linhas, COUNT(DISTINCT sk_movie_id) AS filmes FROM dim_reviews
UNION ALL
SELECT 'movie_reviews', COUNT(*), COUNT(DISTINCT sk_movie_id) FROM movie_reviews
"""))
q("""
SELECT m.titulo, d.qtd_avaliacoes_usuarios, d.nota_media_usuarios,
       r.n_reviews_texto, ROUND(r.media_rating, 2) AS media_rating_texto
FROM dim_reviews d
JOIN dim_movies m USING (sk_movie_id)
LEFT JOIN (SELECT sk_movie_id, COUNT(*) AS n_reviews_texto, AVG(rating) AS media_rating
           FROM movie_reviews GROUP BY sk_movie_id) r USING (sk_movie_id)
WHERE r.n_reviews_texto IS NOT NULL
ORDER BY r.n_reviews_texto DESC LIMIT 10
""")

,tabela,linhas,filmes
0,dim_reviews,40267,40267
1,movie_reviews,43666,40267


,titulo,qtd_avaliacoes_usuarios,nota_media_usuarios,n_reviews_texto,media_rating_texto
0,Die Hart 2: Die Harter,13,4.99,13,4.99
1,Die Hart 2: Die Harter,12,6.49,12,6.49
2,Die Hart: Die Harter,11,5.56,11,5.56
3,Die Hart: Die Harter,10,5.97,10,5.97
4,Die Hart: Die Harter,10,4.04,10,4.04
5,Die Hart: Die Harter,10,4.45,10,4.45
6,Die Hart 2: Die Harter,10,5.42,10,5.42
7,Die Hart: Die Harter,9,6.49,9,6.49
8,Die Hart 2: Die Harter,9,5.04,9,5.04
9,Duro De Atuar 2,9,6.11,9,6.11


In [20]:
q("""
SELECT COUNT(*) AS filmes_nas_duas,
       SUM(d.qtd_avaliacoes_usuarios = r.n) AS qtd_bate,
       SUM(ABS(d.nota_media_usuarios - r.media) < 0.05) AS media_bate
FROM dim_reviews d
JOIN (SELECT sk_movie_id, COUNT(*) AS n, AVG(rating) AS media FROM movie_reviews GROUP BY sk_movie_id) r
  USING (sk_movie_id)
""")

,filmes_nas_duas,qtd_bate,media_bate
0,40267,40267,40267


### 4.8 “Filmes mais avaliados”: qual métrica usar?
Comparamos o top 5 por votos TMDB, votos IMDb, avaliações de usuários (`dim_reviews`) e número de reviews com texto (`movie_reviews`).

In [21]:
tops = {
    "qtd_tmdb": "SELECT m.titulo, f.qtd_tmdb AS valor FROM fact_movies_performance f JOIN dim_movies m USING (sk_movie_id) ORDER BY f.qtd_tmdb DESC LIMIT 5",
    "qtd_imdb": "SELECT m.titulo, f.qtd_imdb AS valor FROM fact_movies_performance f JOIN dim_movies m USING (sk_movie_id) ORDER BY f.qtd_imdb DESC LIMIT 5",
    "dim_reviews.qtd_avaliacoes_usuarios": "SELECT m.titulo, d.qtd_avaliacoes_usuarios AS valor FROM dim_reviews d JOIN dim_movies m USING (sk_movie_id) ORDER BY d.qtd_avaliacoes_usuarios DESC LIMIT 5",
    "movie_reviews (contagem)": "SELECT m.titulo, COUNT(*) AS valor FROM movie_reviews r JOIN dim_movies m USING (sk_movie_id) GROUP BY r.sk_movie_id ORDER BY valor DESC LIMIT 5",
}
for nome, sql in tops.items():
    print(f"Top 5 por {nome}")
    display(q(sql))

Top 5 por qtd_tmdb


,titulo,valor
0,Deadpool,28894
1,Avengers: Infinity War,27713
2,Avengers: Endgame,23857
3,Joker,23425
4,Captain America: Civil War,21541


Top 5 por qtd_imdb


,titulo,valor
0,Stranger Things,1735566
1,Stranger Things,1735566
2,Joker,1723035
3,Avengers: Endgame,1484150
4,Avengers: Infinity War,1406782


Top 5 por dim_reviews.qtd_avaliacoes_usuarios


,titulo,valor
0,Die Hart 2: Die Harter,13
1,Die Hart 2: Die Harter,12
2,Die Hart: Die Harter,11
3,Die Hart: Die Harter,10
4,Die Hart: Die Harter,10


Top 5 por movie_reviews (contagem)


,titulo,valor
0,Die Hart 2: Die Harter,13
1,Die Hart 2: Die Harter,12
2,Die Hart: Die Harter,11
3,Die Hart: Die Harter,10
4,Die Hart: Die Harter,10


### 4.9 Data de lançamento
Qual o tipo/formato de `data_lancamento`, o período coberto e se `ano_lancamento` é coerente com a data?

In [22]:
datas = q("SELECT data_lancamento, ano_lancamento FROM dim_movies")
print("tipos SQLite:", q("SELECT typeof(data_lancamento) AS t, COUNT(*) AS n FROM dim_movies GROUP BY 1").values.tolist())
print("nulos/vazios:", (datas["data_lancamento"].isna() | (datas["data_lancamento"].astype(str).str.strip() == "")).sum())


def padrao(v: object) -> str:
    """Troca dígitos por 9 para revelar o formato da string."""
    return "NULO" if v is None else re.sub(r"\d", "9", str(v))


display(datas["data_lancamento"].map(padrao).value_counts().to_frame("qtd"))
display(datas.sample(5, random_state=1))
q("""
SELECT MIN(data_lancamento) AS mais_antiga, MAX(data_lancamento) AS mais_recente,
       MIN(ano_lancamento) AS menor_ano, MAX(ano_lancamento) AS maior_ano,
       SUM(CAST(substr(data_lancamento, 1, 4) AS INTEGER) <> ano_lancamento) AS ano_divergente
FROM dim_movies WHERE data_lancamento IS NOT NULL
""")

tipos SQLite: [['text', 95645]]
nulos/vazios: 0


,qtd
data_lancamento,
9999-99-99,95645


,data_lancamento,ano_lancamento
38693,2018-06-16,2018
75143,2022-04-15,2022
9746,2018-02-22,2018
54816,2020-10-25,2020
66790,2018-01-01,2018


,mais_antiga,mais_recente,menor_ano,maior_ano,ano_divergente
0,2016-01-01,2029-10-13,2016,2029,0


In [23]:
display(q("SELECT titulo, data_lancamento, status_filme FROM dim_movies WHERE data_lancamento IS NOT NULL ORDER BY data_lancamento LIMIT 3"))
display(q("SELECT titulo, data_lancamento, status_filme FROM dim_movies ORDER BY data_lancamento DESC LIMIT 3"))
created = q("SELECT created_at FROM movie_reviews")["created_at"]
print("movie_reviews.created_at — formatos:")
display(created.map(padrao).value_counts().head().to_frame("qtd"))
print("min:", created.min(), "| max:", created.max())

,titulo,data_lancamento,status_filme
0,Yosemite,2016-01-01,Lançado
1,The Energy Specialist,2016-01-01,Lançado
2,Rosamunde Pilcher: Argentinischer Tango,2016-01-01,Lançado


,titulo,data_lancamento,status_filme
0,Indiana Jones: Return Of Elsa Schneider,2029-10-13,Planejado
1,শূন্যতা,2027-04-21,Planejado
2,Pela Adhi Akshar,2027-03-24,Planejado


movie_reviews.created_at — formatos:


,qtd
created_at,
9999-99-99 99:99:99,43666


min: 2026-09-28 15:49:34 | max: 2026-09-28 15:49:34


### 4.10 Sinopse
Existe sinopse em `dim_movies`? Quantas estão vazias e qual o tamanho típico?

In [24]:
q("""
SELECT COUNT(*) AS total,
       SUM(sinopse IS NULL) AS nulas,
       SUM(TRIM(COALESCE(sinopse,'')) = '') AS nulas_ou_vazias,
       ROUND(AVG(LENGTH(sinopse)), 0) AS tamanho_medio,
       MAX(LENGTH(sinopse)) AS tamanho_max
FROM dim_movies
""")

,total,nulas,nulas_ou_vazias,tamanho_medio,tamanho_max
0,95645,0,0,227.0,1005


In [25]:
q("SELECT titulo, sinopse FROM dim_movies WHERE sinopse IS NOT NULL AND TRIM(sinopse) <> '' LIMIT 3")

,titulo,sinopse
0,Rings,"""Julia becomes worried about her boyfriend Holt when he explores the dark ur..."
1,Mixtape,"On the eve of Y2K, orphaned 12-year-old Beverly discovers a broken mixtape c..."
2,Grizzly Ii: Revenge,"""All hell breaks loose when a giant grizzly, reacting to the slaughter of he..."


### 4.11 Gêneros
Valores distintos de gênero e quantos filmes cada um tem (um filme pode ter vários gêneros).

In [26]:
q("""
SELECT g.sk_genre_id, g.nome_genero, COUNT(b.sk_movie_id) AS filmes
FROM dim_genres g LEFT JOIN bridge_movie_genre b USING (sk_genre_id)
GROUP BY g.sk_genre_id, g.nome_genero
ORDER BY filmes DESC
""")

,sk_genre_id,nome_genero,filmes
0,da72bf5bbca4ca33b822955cfbbf9e5b84e4e0fa5e5e68cf9d30054109fac58b,Drama,28086
1,43f1fc2390a440781ffc1359e14ecb8389bfaaf8319bbe09f291a3872ea49bb9,Documentary,18082
2,85f1c8c8e324b6be99b13732edd1770eb0d200d15becbd659cc47ff5e060ac43,Comedy,16048
3,271a22f75d73e13fa558d991442ecde8d75a34a0b9f6216d44b116f8612c82dc,Horror,8674
4,9360404e4f35ac209099b0fce8615971c8e89d2ab06348841d708bc3f20d28e3,Thriller,8540
5,3bc3567a7fc6511a85072520dc1bdb95a732c4aee93a6533d53954f39124d7c9,Romance,6209
6,64cff1319d2fd2cbb7a1e84ccecf22c1cc07b24435cdb522f8c0aa525d6002a6,Action,5028
7,1dd10f244a62771bcca9df74a9bc7d13363df8d3cd69ee07f66c2b1666b256ff,Animation,3911
8,22611ceccd0b4d14703ccd4eaab3208ffc03d2c80d38655023b622e50d0466fe,Crime,3902
9,852a56bfaf803f454ce5e6380225fe1770c1e567bbb32cf335c99385c3930025,Tv Movie,3336


In [27]:
q("""
SELECT n_generos, COUNT(*) AS filmes FROM (
  SELECT m.sk_movie_id, COUNT(b.sk_genre_id) AS n_generos
  FROM dim_movies m LEFT JOIN bridge_movie_genre b USING (sk_movie_id) GROUP BY m.sk_movie_id
) GROUP BY n_generos ORDER BY n_generos
""")

,n_generos,filmes
0,0,20037
1,1,44280
2,2,19840
3,3,8987
4,4,2034
5,5,369
6,6,77
7,7,16
8,8,2
9,9,2


### 4.12 Extras: status e idioma original
Valores categóricos que o usuário pode usar em filtros.

In [28]:
display(q("SELECT status_filme, COUNT(*) AS filmes FROM dim_movies GROUP BY 1 ORDER BY 2 DESC"))
q("SELECT idioma_original, COUNT(*) AS filmes FROM dim_movies GROUP BY 1 ORDER BY 2 DESC LIMIT 10")

,status_filme,filmes
0,Lançado,94304
1,Pós-Produção,693
2,Em Produção,598
3,Planejado,50


,idioma_original,filmes
0,None,95645


## 5. Achados extras (surgiram durante a exploração)

### 5.1 A taxa BRL/USD varia — depende do ano?
A razão BRL/USD não é constante (3,05 a 5,78). Verificamos se é uma taxa por ano de lançamento.

In [29]:
q("""
SELECT m.ano_lancamento,
       ROUND(MIN(f.receita_brl * 1.0 / f.receita_usd), 4) AS taxa_min,
       ROUND(MAX(f.receita_brl * 1.0 / f.receita_usd), 4) AS taxa_max,
       COUNT(*) AS filmes
FROM fact_movies_performance f JOIN dim_movies m USING (sk_movie_id)
WHERE f.receita_usd > 0
GROUP BY m.ano_lancamento ORDER BY m.ano_lancamento
""")

,ano_lancamento,taxa_min,taxa_max,filmes
0,2016,3.1187,4.1552,576
1,2017,3.0504,3.3756,588
2,2018,3.1380,4.1873,511
3,2019,3.6513,4.2596,490
4,2020,4.0207,5.7797,308
5,2021,4.9200,5.8391,332
6,2022,4.6391,5.7036,357
7,2023,4.7247,5.4453,199
8,2024,4.8537,5.6618,12


### 5.2 Popularidade com valor igual ao ano
No top 5 apareceram popularidades 2020, 2019 e 2018 em filmes com poucos votos — parece erro de carga.

In [30]:
display(q("""
SELECT COUNT(*) AS popularidade_igual_ao_ano
FROM fact_movies_performance f JOIN dim_movies m USING (sk_movie_id)
WHERE f.popularidade = m.ano_lancamento
"""))
q("""
SELECT m.titulo, m.ano_lancamento, f.popularidade, f.qtd_tmdb
FROM fact_movies_performance f JOIN dim_movies m USING (sk_movie_id)
WHERE f.popularidade > 100
ORDER BY f.popularidade DESC LIMIT 15
""")

,popularidade_igual_ao_ano
0,2


,titulo,ano_lancamento,popularidade,qtd_tmdb
0,Blue Beetle,2023,2994.357,1023.0
1,Gran Turismo,2023,2680.593,702.0
2,La Fellinette,2020,2020.000,2.0
3,The Fear Footage 2: Curse Of The Tape,2020,2019.000,5.0
4,Wwe Survivor Series 2018,2018,2018.000,17.0
5,Battipaglia 1969,2016,1969.000,1.0
6,The Nun Ii,2023,1692.778,NaN
7,Meg 2: The Trench,2023,1567.273,2034.0
8,Retribution,2023,1547.220,258.0
9,Talk To Me,2023,1458.514,973.0


### 5.3 Títulos repetidos
`Die Hart` e `Stranger Things` apareceram mais de uma vez nos rankings. Quantos títulos se repetem com `sk_movie_id` diferentes?

In [31]:
display(q("""
SELECT COUNT(*) AS titulos_repetidos, SUM(n) AS filmes_envolvidos
FROM (SELECT LOWER(titulo) AS t, COUNT(*) AS n FROM dim_movies GROUP BY 1 HAVING COUNT(*) > 1)
"""))
q("""
SELECT m.titulo, m.id_filme, m.ano_lancamento, m.duracao_minutos, f.qtd_tmdb, f.nota_imdb, f.qtd_imdb
FROM dim_movies m JOIN fact_movies_performance f USING (sk_movie_id)
WHERE m.titulo IN ('Stranger Things', 'Die Hart: Die Harter')
ORDER BY m.titulo, m.ano_lancamento
""")

,titulos_repetidos,filmes_envolvidos
0,4558,12165


,titulo,id_filme,ano_lancamento,duracao_minutos,qtd_tmdb,nota_imdb,qtd_imdb
0,Die Hart: Die Harter,1300214,2024,92,2,4.3,6181.0
1,Die Hart: Die Harter,1305205,2024,92,0,4.3,6181.0
2,Die Hart: Die Harter,1314611,2024,92,0,4.3,6181.0
3,Die Hart: Die Harter,1322058,2024,0,0,4.3,6181.0
4,Die Hart: Die Harter,1336604,2024,0,0,4.3,6181.0
5,Die Hart: Die Harter,1357515,2024,92,0,4.3,NaN
6,Die Hart: Die Harter,1392978,2024,92,0,4.3,6181.0
7,Die Hart: Die Harter,1418800,2024,0,1,4.3,6181.0
8,Die Hart: Die Harter,1423077,2024,0,0,4.3,NaN
9,Die Hart: Die Harter,1426502,2024,0,0,4.3,6181.0


### 5.4 Duração e idioma
`duracao_minutos` tem zeros e máximo de 13.319 min; `idioma_original` parecia todo nulo.

In [32]:
q("""
SELECT SUM(duracao_minutos = 0) AS duracao_zero,
       SUM(duracao_minutos > 300) AS duracao_maior_300,
       SUM(idioma_original IS NULL) AS idioma_nulo,
       COUNT(*) AS total
FROM dim_movies
""")

,duracao_zero,duracao_maior_300,idioma_nulo,total
0,10160,107,95645,95645


Fechamos a conexão.

In [33]:
conn.close()

## Conclusões

### O que foi descoberto

**Estrutura**
- 10 tabelas de negócio em esquema estrela (+ `alembic_version`, que é só controle de migração). Centro: `dim_movies` (95.645 filmes); `fact_movies_performance` tem exatamente 1 linha por filme.
- Todas as chaves são hashes `VARCHAR(64)` (`sk_*`). As chaves estrangeiras estão declaradas e **não há órfãos** em nenhuma ligação.
- Ligações: `dim_movies.sk_movie_id` → fact, `dim_reviews`, `movie_reviews` e as 3 bridges; `bridge_movie_genre` → `dim_genres`, `bridge_movie_person` → `dim_people`, `bridge_movie_company` → `dim_companies`.

**Dinheiro**
- Receita e orçamento **nunca são zero, só nulos** quando não informados: receita informada em 3.373 filmes (3,5%), orçamento em 7.926 (8,3%), ambos em só **1.630** (1,7%).
- `lucro` nunca é nulo: é sempre `COALESCE(receita,0) − COALESCE(orcamento,0)` (0 divergências em USD e BRL). Por isso:
  - 6.296 filmes têm orçamento, mas não têm receita, e ficam com lucro = −orçamento (ex.: *Enea*, −888 mi; *The Batman*, −185 mi);
  - 1.743 filmes têm receita, mas não têm orçamento, e ficam com lucro = receita (inflado);
  - 86.032 filmes sem nenhum dos dois ficam com lucro 0.
- Há colunas em **R$** (`orcamento_brl`, `receita_brl`, `lucro_brl`), além das em US$. A taxa BRL/USD **não é fixa**: varia de 3,05 a 5,84 e acompanha o câmbio da época do lançamento (≈3,1–3,4 em 2017, ≈4,9–5,8 em 2021).

**Notas e avaliações**
- `nota_tmdb` e `nota_imdb` estão na escala **0–10**. `nota_tmdb = 0` em 36.185 filmes, e quase todos (36.043) não têm votos, então 0 significa "sem nota". `nota_imdb` usa NULL para "sem nota" (12.674 filmes).
- `qtd_tmdb` / `qtd_imdb` = número de votos. Os dados do IMDb parecem ter sido ligados **pelo título**: as 24 cópias de *Die Hart: Die Harter* têm o mesmo `qtd_imdb` (6.181), e *Stranger Things* (uma série) aparece 2× com 1,7 mi de votos.
- `dim_reviews` = **agregado por filme** (1 linha por filme, `sk_review_id = sk_movie_id`): `qtd_avaliacoes_usuarios` e `nota_media_usuarios`. `movie_reviews` = **reviews individuais** (autor, nota 0–10, texto em PT, data). As duas batem 100%: a contagem e a média de `movie_reviews` são iguais às de `dim_reviews` nos 40.267 filmes. As reviews parecem sintéticas: todas foram criadas em 2026-09-28 15:49:34 e o máximo é 13 reviews por filme.
- "Mais avaliados": o top por `qtd_tmdb` traz filmes conhecidos (*Deadpool*, *Avengers*, *Joker*). O top por `qtd_imdb` traz a série *Stranger Things*. Os tops por reviews de usuários trazem duplicatas de *Die Hart* com 10–13 reviews.

**Popularidade**
- Coluna `popularidade` (índice TMDB, sem limite superior): 3.615 nulos, 1.698 zeros, mediana 0,92, p99 30,7, máximo 2.994. É muito assimétrica.
- Há ruído: valores como 2020, 2019, 2018 e 1969 em filmes com 1–17 votos (número tirado do título ou do ano).

**Pessoas**
- `bridge_movie_person` só tem chaves. O papel está em **`dim_people.tipo_pessoa`**, com valores `Ator`, `Diretor` e `Roteirista`.
- A mesma pessoa tem **um registro por papel**: 48.210 nomes aparecem com os 3 tipos, cada um com um `sk_person_id` diferente. Também há "nomes" lixo, como `0.6` e `1.4`.

**Datas, textos e categorias**
- `data_lancamento` é TEXT no formato ISO `AAAA-MM-DD`, sem nulos, de **2016-01-01 a 2029-10-13**. As datas futuras são filmes com status `Planejado`. `ano_lancamento` é sempre coerente com a data.
- `sinopse` existe e está preenchida em 100% dos filmes (média de 227 caracteres), **em inglês**.
- 19 gêneros, com **nomes em inglês** (Drama, Documentary, Comedy, …, Tv Movie, Science Fiction, Western). 20.037 filmes não têm gênero, e um filme pode ter vários.
- `status_filme` está em português: Lançado (94.304), Pós-Produção, Em Produção, Planejado.
- `idioma_original` é **100% nulo**. `duracao_minutos` tem 10.160 zeros (desconhecida) e 107 valores acima de 300 min (máx. 13.319).
- **Títulos repetidos**: 4.558 títulos aparecem mais de uma vez (12.165 filmes), com `id_filme` diferentes. Títulos estão em *Title Case* ("Grizzly Ii").

### Regras de negócio implicadas
1. **Receita/orçamento informado** = coluna `IS NOT NULL` (equivale a `> 0`). Em médias, rankings e somas de receita ou orçamento, filtrar os nulos e avisar quantos filmes têm o dado.
2. **Lucro só é válido quando receita e orçamento estão informados** (`receita_usd IS NOT NULL AND orcamento_usd IS NOT NULL`, 1.630 filmes). Nunca ranquear lucro sem esse filtro.
3. **Moeda**: o padrão é US$ (`_usd`). Se o usuário falar em reais/R$, usar as colunas `_brl` (conversão já feita na base, com câmbio histórico) e não converter manualmente.
4. **Notas**: TMDB e IMDb em escala 0–10. Tratar `nota_tmdb = 0` com `qtd_tmdb = 0` como sem nota, e exigir um mínimo de votos em rankings de "melhores" (ex.: `qtd_tmdb >= 100`). Deixar claro qual fonte foi usada (TMDB, IMDb ou usuários).
5. **"Mais avaliados / mais votados"**: usar `qtd_tmdb` por padrão (é por filme e confiável). `qtd_imdb` tem ligação suspeita por título. Para "reviews/avaliações de usuários", usar `dim_reviews` (agregado). Usar `movie_reviews` só para o texto das reviews ou para filtros por autor e data.
6. **Popularidade**: usar `fact_movies_performance.popularidade`, ignorar nulos e, em rankings, exigir votos mínimos para filtrar o ruído.
7. **Ator, diretor e roteirista**: sempre filtrar `dim_people.tipo_pessoa IN ('Ator','Diretor','Roteirista')` no join com `bridge_movie_person`. Ao buscar uma pessoa pelo nome, lembrar que ela tem um registro por papel.
8. **Gêneros**: traduzir o gênero pedido em português para o nome em inglês da base (Ação → Action, Comédia → Comedy, Ficção científica → Science Fiction, Terror → Horror, Documentário → Documentary etc.). Filmes podem ter vários gêneros.
9. **Datas**: filtrar por ano com `ano_lancamento`. Para períodos, comparar `data_lancamento` como texto ISO. A base cobre 2016–2029, e o agente deve avisar quando a pergunta sair desse intervalo. "Filmes lançados" = `status_filme = 'Lançado'`.
10. **Títulos**: buscar com comparação sem diferença de maiúsculas (`LIKE` / `LOWER`) e tratar títulos repetidos (mostrar ano/`id_filme` ou deduplicar).
11. **Campos inutilizáveis**: `idioma_original` (100% nulo), que o agente deve declarar como indisponível. Em `duracao_minutos`, desconsiderar 0 e tratar valores muito altos com cautela.
12. **Sinopse** está em inglês: buscas por tema no texto devem usar palavras em inglês.
